# APIM ❤️ AI Agents

## Federated MCP with API Management Workspaces lab
![flow](../../images/a2a-mcp-workspaces.gif)

This lab demonstrates a **federated** AI gateway built on Azure API Management **workspaces** (Standard v2).

Two decentralized API teams each own their tools in an isolated **workspace**, associated with the service's **default managed gateway**:

- **Weather team** → a team-owned **Weather API** (in the `weather-ws` workspace)
- **OnCall team** → a team-owned **OnCall API** (in the `oncall-ws` workspace)

Each workspace API forwards to a skeleton backend service hosted on **Azure Container Apps** (built from [src/weather/app](src/weather/app) and [src/oncall/app](src/oncall/app)), so the tools are backed by real APIs rather than mocked responses.

The central API platform team exposes each workspace API as a **Model Context Protocol (MCP) server** at the **service level** (MCP servers aren't supported inside workspaces, so they proxy to the workspace-owned APIs). A shared, service-level **Inference API** is available to all agents.

Models are centralized using the **Model Gateway** pattern across three Azure AI Foundry instances:

- **`inference-foundry`** hosts the model deployments and is exposed as a shared, service-level endpoint through the APIM Inference API.
- **`weather-foundry`** and **`oncall-foundry`** host **no models** of their own. Each reaches the models through an APIM **model-gateway connection** (`ai-gateway`) and hosts its team's agent.

Finally, two **Azure AI Foundry agents** consume these MCP servers — a Weather agent (on `weather-foundry`) and an OnCall agent (on `oncall-foundry`) — showing how federated, team-owned APIs *and* centralized models are both governed through a single gateway.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles

- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

- [Docker](https://www.docker.com/) is **not** required — the container images are built remotely with `az acr build`.▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according to your preferences and to [product availability by Azure region](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management).
- **Workspaces require the Basic v2, Standard v2, Premium, or Premium v2 tier.** This lab uses `Standardv2`.

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

subscription_id = utils.get_current_subscription()
deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"  # change to match your naming style or an existing resource group
resource_group_location = "australiaeast" #norwayeast

# AI Foundry instances (Model Gateway pattern):
#   - inference-foundry : hosts the model deployments; exposed as a shared service-level endpoint via APIM
#   - weather-foundry   : no models; reaches models through an APIM model-gateway connection. Hosts the Weather agent.
#   - oncall-foundry    : no models; reaches models through an APIM model-gateway connection. Hosts the OnCall agent.
# The "weight" routes inference traffic only to the inference foundry (team foundries stay at weight 0).
aiservices_config = [
    {"name": "inference-foundry", "location": resource_group_location, "weight": 1},
    {"name": "weather-foundry", "location": resource_group_location, "weight": 0},
    {"name": "oncall-foundry", "location": resource_group_location, "weight": 0},
]
# Models are deployed only to the inference foundry (targeted via the "aiservice" field)
models_config = [{"name": "gpt-4.1-mini", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 20, "aiservice": "inference-foundry"}]

# Name of the model-gateway connection created on each team Foundry instance
model_gateway_connection_name = "ai-gateway"

# API Management - Standard v2 is required for workspaces
apim_sku = 'Premiumv2'  # options: Developer, Basic, Standard, Premium
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# Shared, service-level Inference API
inference_api_path = "inference"          # path to the inference API in the APIM service
inference_api_type = "PassThrough"        # options: AzureOpenAI, AzureAI, PassThrough
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

# True APIM workspaces (associated with the default managed gateway)
weather_workspace = {"name": "weather-ws", "displayName": "Weather Team Workspace"}
oncall_workspace = {"name": "oncall-ws", "displayName": "OnCall Team Workspace"}

# Skeleton backend APIs hosted on Azure Container Apps
weather_app_src = "src/weather/app"   # Dockerfile + FastAPI app for the Weather backend
oncall_app_src = "src/oncall/app"     # Dockerfile + FastAPI app for the OnCall backend
weather_image = "weather-api"         # image repository name in ACR
oncall_image = "oncall-api"           # image repository name in ACR

utils.print_ok('Notebook initialized')


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

The template deploys API Management (Standard v2), three Azure AI Foundry instances (an `inference-foundry` that hosts the model deployment plus a `weather-foundry` and `oncall-foundry` that access models through an APIM model-gateway connection), a shared service-level Inference API, an Azure Container Registry and Container Apps environment hosting the two skeleton backend APIs, two workspaces (each with a team-owned API that forwards to its container app), and a service-level MCP server per workspace API.


In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "modelGatewayConnectionName": { "value": model_gateway_connection_name },
        "weatherWorkspaceName": { "value": weather_workspace["name"] },
        "weatherWorkspaceDisplayName": { "value": weather_workspace["displayName"] },
        "oncallWorkspaceName": { "value": oncall_workspace["name"] },
        "oncallWorkspaceDisplayName": { "value": oncall_workspace["displayName"] },
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, subscription key, Foundry project endpoint and the MCP server endpoints.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    apim_service_name = utils.get_deployment_output(output, 'apimServiceName', 'APIM Service Name')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    #workspace_gateway_url = utils.get_deployment_output(output, 'workspaceGatewayUrl', 'Workspace Gateway URL')

    # Foundry instances: inference (hosts models) + one per team (model gateway consumers)
    inference_foundry_project_endpoint = utils.get_deployment_output(output, 'inferenceFoundryProjectEndpoint', 'Inference Foundry Project Endpoint')
    weather_foundry_project_endpoint = utils.get_deployment_output(output, 'weatherFoundryProjectEndpoint', 'Weather Foundry Project Endpoint')
    oncall_foundry_project_endpoint = utils.get_deployment_output(output, 'oncallFoundryProjectEndpoint', 'OnCall Foundry Project Endpoint')
    model_gateway_connection_name = utils.get_deployment_output(output, 'modelGatewayConnectionName', 'Model Gateway Connection Name')

    weather_mcp_endpoint = utils.get_deployment_output(output, 'weatherMcpEndpoint', 'Weather MCP Endpoint')
    oncall_mcp_endpoint = utils.get_deployment_output(output, 'oncallMcpEndpoint', 'OnCall MCP Endpoint')
    weather_api_path = utils.get_deployment_output(output, 'weatherApiPath', 'Weather API Path')
    oncall_api_path = utils.get_deployment_output(output, 'oncallApiPath', 'OnCall API Path')

    # Azure Container Apps hosting the skeleton backend APIs
    container_registry_name = utils.get_deployment_output(output, 'containerRegistryName', 'Container Registry Name')
    weather_app_name = utils.get_deployment_output(output, 'weatherAppName', 'Weather Container App')
    oncall_app_name = utils.get_deployment_output(output, 'oncallAppName', 'OnCall Container App')
    weather_app_url = utils.get_deployment_output(output, 'weatherAppUrl', 'Weather Container App URL')
    oncall_app_url = utils.get_deployment_output(output, 'oncallAppUrl', 'OnCall Container App URL')

    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        utils.print_info(f"Subscription Name: {subscription['name']}")
        utils.print_info(f"Subscription Key: ****{subscription['key'][-4:]}")
    api_key = apim_subscriptions[0].get("key")


<a id='4'></a>
### 4️⃣ Build and deploy the skeleton backend APIs to Azure Container Apps

The Bicep deployment created the container platform (Azure Container Registry + Container Apps environment) and two container apps with a placeholder image. Here we build the skeleton **Weather** and **OnCall** backend images from [src/weather/app](src/weather/app) and [src/oncall/app](src/oncall/app), push them to the registry, and roll the container apps onto the real images.

Each workspace-owned API in API Management forwards to its container app, so the tools are backed by real services rather than mocked responses.


In [ ]:
build = 1  # increment to build and roll out a new image version

# Build and push the skeleton backend images to the Azure Container Registry
utils.run(f"az acr build --image {weather_image}:v0.{build} --resource-group {resource_group_name} --registry {container_registry_name} --file {weather_app_src}/Dockerfile {weather_app_src}/. --no-logs",
    "Weather backend image built and pushed", "Failed to build the Weather backend image")

utils.run(f"az acr build --image {oncall_image}:v0.{build} --resource-group {resource_group_name} --registry {container_registry_name} --file {oncall_app_src}/Dockerfile {oncall_app_src}/. --no-logs",
    "OnCall backend image built and pushed", "Failed to build the OnCall backend image")

# Roll the container apps onto the freshly built images
utils.run(f'az containerapp update -n {weather_app_name} -g {resource_group_name} --image "{container_registry_name}.azurecr.io/{weather_image}:v0.{build}"',
    "Weather container app updated", "Failed to update the Weather container app")

utils.run(f'az containerapp update -n {oncall_app_name} -g {resource_group_name} --image "{container_registry_name}.azurecr.io/{oncall_image}:v0.{build}"',
    "OnCall container app updated", "Failed to update the OnCall container app")


<a id='5'></a>
### 5️⃣ Associate the workspaces with the default managed gateway

Workspace APIs need a gateway to run. In the v2 tiers you can associate a workspace with the service's **default managed gateway** (`serveOn: workspaceAndDefault`) instead of provisioning a separate workspace gateway (which can take hours and adds cost). This association is currently set through the API Management REST API, so we apply it here after the Bicep deployment.

Once associated, each workspace API is reachable on the default hostname (for example, `https://<service>.azure-api.net/weather/current`).


In [ ]:
def associate_workspace_with_default_gateway(ws):
    url = (f"https://management.azure.com/subscriptions/{subscription_id}"
           f"/resourceGroups/{resource_group_name}/providers/Microsoft.ApiManagement"
           f"/service/{apim_service_name}/workspaces/{ws['name']}?api-version=2025-09-01-preview")
    body = {"properties": {"displayName": ws["displayName"], "serveOn": "workspaceAndDefault"}}
    with open('workspace-body.json', 'w') as f:
        json.dump(body, f)
    return utils.run(f"az rest --method put --url \"{url}\" --body @workspace-body.json",
                     f"Workspace '{ws['name']}' associated with the default managed gateway",
                     f"Failed to associate workspace '{ws['name']}'")

for ws in [weather_workspace, oncall_workspace]:
    associate_workspace_with_default_gateway(ws)

In [ ]:
for ws in [weather_workspace, oncall_workspace]:
    url = (f"https://management.azure.com/subscriptions/{subscription_id}"
           f"/resourceGroups/{resource_group_name}/providers/Microsoft.ApiManagement"
           f"/service/{apim_service_name}/workspaces/{ws['name']}?api-version=2025-09-01-preview")
    output = utils.run(f'az rest --method get --url "{url}"', ...)
    if output.success and output.json_data:
        print(json.dumps(output.json_data, indent=2))

<a id='6'></a>
### 6️⃣ Test the workspace-owned APIs proxied

Call each team-owned API on the default gateway hostname. Each API forwards to its **Azure Container Apps** backend. Gateway configuration can take a short while to propagate after the workspace association, so we retry until the API responds.


In [ ]:
import time, requests

def call_with_retry(url, params, attempts=10, delay=15):
    response = None
    for i in range(attempts):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            return response
        utils.print_info(f"Attempt {i+1}/{attempts}: status {response.status_code}. Waiting for gateway propagation...")
        time.sleep(delay)
    return response

# Weather API
weather_response = call_with_retry(f"{apim_resource_gateway_url}/{weather_api_path}-tool/current", {"city": "Lisbon"})
if weather_response.status_code == 200:
    utils.print_ok("Proxied Weather workspace API responded")
    print(json.dumps(weather_response.json(), indent=2))
else:
    utils.print_error(f"Weather workspace API failed: {weather_response.status_code} - {weather_response.text}")

# # OnCall API
# oncall_response = call_with_retry(f"https://workspace-gateway-c0f5ahfva5dpfth3.gateway.australiaeast-01.azure-api.net/{oncall_api_path}/status", {"team": "platform"})
# if oncall_response.status_code == 200:
#     utils.print_ok("Direct OnCall workspace API responded")
#     print(json.dumps(oncall_response.json(), indent=2))
# else:
#     utils.print_error(f"OnCall workspace API failed: {oncall_response.status_code} - {oncall_response.text}")

oncall_response = call_with_retry(f"{apim_resource_gateway_url}/{oncall_api_path}-tool/status", {"team": "platform"})
if oncall_response.status_code == 200:
    utils.print_ok("Proxied OnCall workspace API responded")
    print(json.dumps(oncall_response.json(), indent=2))
else:
    utils.print_error(f"OnCall workspace API failed: {oncall_response.status_code} - {oncall_response.text}")

<a id='7'></a>
### 7️⃣ Weather agent — consume the Weather MCP server - via shared MCP services

Create an [Azure AI Foundry agent](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/model-context-protocol) on the **`weather-foundry`** instance that uses the **Weather MCP** server exposed by API Management. The MCP server proxies to the Weather API owned by the `weather-ws` workspace, which in turn calls the Weather backend running on Azure Container Apps. Since `weather-foundry` hosts no models, the agent reaches the model through the APIM **model-gateway connection** using the `{connection}/{model}` format.


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# The Weather agent is hosted on the Weather team's Foundry instance, which has no models of its own.
# It reaches the model through the APIM model-gateway connection using the "{connection}/{model}" format.
weather_project_client = AIProjectClient(endpoint=weather_foundry_project_endpoint, credential=credential)
gateway_model = f"{model_gateway_connection_name}/{models_config[0]['name']}"

# MCP tool pointing to the Weather MCP server exposed by APIM
weather_mcp_tool = MCPTool(
    server_label="weather",
    server_url=weather_mcp_endpoint,
    require_approval="never",
)

weather_agent = weather_project_client.agents.create_version(
    agent_name="weather-agent",
    definition=PromptAgentDefinition(
        model=gateway_model,
        instructions=(
            "You are a helpful weather assistant. Use the Weather MCP tools to answer "
            "questions about the current weather. Always mention the city and the temperature."
        ),
        tools=[weather_mcp_tool],
    ),
)
utils.print_ok(f"Agent '{weather_agent.name}' created (version: {weather_agent.version})")
utils.print_info(f"Hosted on Weather Foundry, model via gateway: {gateway_model}")
utils.print_info(f"MCP Server: {weather_mcp_tool.server_label} at {weather_mcp_tool.server_url}")

weather_conversations_client = weather_project_client.get_openai_client()
conversation = weather_conversations_client.conversations.create()

prompt = "What's the current weather in Lisbon and Seattle?"
utils.print_info(f"Question: {prompt}")

response = weather_conversations_client.responses.create(
    conversation=conversation.id,
    tool_choice="required",
    input=prompt,
    extra_body={"agent_reference": {"name": weather_agent.name, "type": "agent_reference", "version": weather_agent.version}},
    stream=False,
)
utils.print_ok(f"Answer:\n{response.output_text}")


<a id='8'></a>
### 8️⃣ OnCall agent — consume the OnCall MCP server - via shared MCP services

Create a second Foundry agent on the **`oncall-foundry`** instance that uses the **OnCall MCP** server. The MCP server proxies to the OnCall API owned by the `oncall-ws` workspace, which calls the OnCall backend running on Azure Container Apps. Each team owns an independent Foundry instance and workspace, yet both share the same governed gateway and the same centralized models served through the model-gateway connection.


In [ ]:
# The OnCall agent is hosted on the OnCall team's Foundry instance (a separate project),
# which also reaches the model through its own APIM model-gateway connection.
oncall_project_client = AIProjectClient(endpoint=oncall_foundry_project_endpoint, credential=credential)

oncall_mcp_tool = MCPTool(
    server_label="oncall",
    server_url=oncall_mcp_endpoint,
    require_approval="never",
)

oncall_agent = oncall_project_client.agents.create_version(
    agent_name="oncall-agent",
    definition=PromptAgentDefinition(
        model=gateway_model,
        instructions=(
            "You are an incident response assistant. Use the OnCall MCP tools to find who is "
            "currently on-call for a team, and include their contact details and escalation policy."
        ),
        tools=[oncall_mcp_tool],
    ),
)
utils.print_ok(f"Agent '{oncall_agent.name}' created (version: {oncall_agent.version})")
utils.print_info(f"Hosted on OnCall Foundry, model via gateway: {gateway_model}")
utils.print_info(f"MCP Server: {oncall_mcp_tool.server_label} at {oncall_mcp_tool.server_url}")

oncall_conversations_client = oncall_project_client.get_openai_client()
conversation = oncall_conversations_client.conversations.create()

prompt = "Who is on-call for the platform and payments teams, and how do I reach them?"
utils.print_info(f"Question: {prompt}")

response = oncall_conversations_client.responses.create(
    conversation=conversation.id,
    tool_choice="required",
    input=prompt,
    extra_body={"agent_reference": {"name": oncall_agent.name, "type": "agent_reference", "version": oncall_agent.version}},
    stream=False,
)
utils.print_ok(f"Answer:\n{response.output_text}")


<a id='9'></a>
### 9️⃣ Publish both agents as A2A endpoints

Expose the Weather and OnCall agents as [Agent2Agent (A2A)](https://a2a-protocol.org/latest/) endpoints so other agents can discover and call them through the A2A protocol. Following [Enable incoming A2A on a Foundry agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint?tabs=python%2Cverify-bash%2Cconnection-bash), we use the Foundry SDK's `patch_agent_details` to set each agent's **agent card** (the discovery metadata other agents see) and add the **`a2a`** protocol to its endpoint.

Once enabled, each agent publishes an authenticated agent card at `.../endpoint/protocols/a2a/agentCard/v1.0` and accepts inbound A2A requests. All A2A URLs require Microsoft Entra ID authentication — callers must present a token with the **Foundry Agent Consumer** role (or higher) on the project.

> **Note:** Incoming A2A is in **preview** and requires the responses protocol (both prompt agents above qualify). This uses `azure-ai-projects>=2.2.0`, where the `AgentEndpointConfig` and `AgentCard` models are exposed through `project_client.beta.agents`.


In [ ]:
from azure.ai.projects.models import (
    AgentEndpointConfig,
    AgentEndpointProtocol,
    AgentCard,
    AgentCardSkill,
)

def publish_agent_as_a2a(project_client, project_endpoint, agent, description, skills):
    # Configure the agent card (discovery metadata) and add the A2A protocol to the endpoint
    project_client.beta.agents.patch_agent_details(
        agent_name=agent.name,
        agent_endpoint=AgentEndpointConfig(
            protocols=[
                AgentEndpointProtocol.RESPONSES,
                AgentEndpointProtocol.A2A,
                AgentEndpointProtocol.MCP,
                AgentEndpointProtocol.ACTIVITY,
                AgentEndpointProtocol.INVOCATIONS,
            ],
        ),
        agent_card=AgentCard(
            version="1.0",
            description=description,
            skills=skills,
        ),
    )
    utils.print_ok(f"Incoming A2A enabled for '{agent.name}'")

    a2a_base = f"{project_endpoint}/agents/{agent.name}/endpoint/protocols/a2a"
    utils.print_info(f"A2A base path: {a2a_base}")
    utils.print_info(f"Agent card (v1.0): {a2a_base}/agentCard/v1.0")

publish_agent_as_a2a(
    weather_project_client,
    weather_foundry_project_endpoint,
    weather_agent,
    "A weather assistant that reports the current conditions for any city.",
    [AgentCardSkill(id="current-weather", name="Current Weather",
                    description="Returns the current weather for a given city.")],
)

publish_agent_as_a2a(
    oncall_project_client,
    oncall_foundry_project_endpoint,
    oncall_agent,
    "An incident response assistant that identifies who is on-call for a team.",
    [AgentCardSkill(id="oncall-lookup", name="On-Call Lookup",
                    description="Finds the current on-call person for a team, their contact details, and escalation policy.")],
)


In [ ]:
import requests

# Microsoft Entra token for the Foundry data plane (reuse the credential from step 7)
a2a_token = credential.get_token("https://ai.azure.com/.default").token
a2a_headers = {"Authorization": f"Bearer {a2a_token}"}

def read_agent_card(project_endpoint, agent):
    # Fetch the published v1.0 agent card via the raw A2A REST endpoint
    agent_card_url = f"{project_endpoint}/agents/{agent.name}/endpoint/protocols/a2a/agentCard/v1.0"
    response = requests.get(agent_card_url, headers=a2a_headers)
    if response.status_code == 200:
        utils.print_ok(f"Agent card for '{agent.name}' (direct to Foundry):")
        print(json.dumps(response.json(), indent=2))
    else:
        utils.print_error(f"Failed to read agent card for '{agent.name}': {response.status_code} - {response.text}")

read_agent_card(weather_foundry_project_endpoint, weather_agent)
read_agent_card(oncall_foundry_project_endpoint, oncall_agent)

In [ ]:
import uuid, time

TERMINAL_A2A_STATES = {"completed", "failed", "canceled", "rejected", "input-required"}

def _jsonrpc(message_url, headers, method, params):
    payload = {"jsonrpc": "2.0", "id": str(uuid.uuid4()), "method": method, "params": params}
    response = requests.post(message_url, headers=headers, json=payload)
    response.raise_for_status()
    body = response.json()
    if "error" in body:
        raise RuntimeError(body["error"])
    return body["result"]

def _extract_text(task):
    # Prefer artifacts, fall back to the last agent message in history
    texts = []
    for artifact in task.get("artifacts", []):
        for part in artifact.get("parts", []):
            if part.get("kind") == "text":
                texts.append(part["text"])
    if not texts:
        for msg in reversed(task.get("history", [])):
            if msg.get("role") == "agent":
                for part in msg.get("parts", []):
                    if part.get("kind") == "text":
                        texts.append(part["text"])
                if texts:
                    break
    return "\n".join(texts)

def invoke_a2a_agent(a2a_base_url, a2a_card, question, poll_attempts=30, poll_delay=2):
    headers = {"Content-Type": "application/json"}

    # Uncomment to enable Microsoft Entra token authentication for the A2A endpoint (requires Foundry v1.0.2025-06-01 or later)
    # token = credential.get_token("https://ai.azure.com/.default").token
    # headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    # Discover the A2A message endpoint from the agent card ("url" field), fall back to the base
    card = requests.get(a2a_card, headers=headers)
    card.raise_for_status()
    message_url = card.json().get("url", a2a_base_url)

    utils.print_info(f"Invoking A2A agent: {message_url}")
    utils.print_info(f"Question: {question}")

    # Send the message (A2A JSON-RPC 2.0 "message/send")
    result = _jsonrpc(message_url, headers, "message/send", {
        "message": {
            "kind": "message",
            "role": "user",
            "parts": [{"kind": "text", "text": question}],
            "messageId": str(uuid.uuid4()),
        }
    })

    # A direct "message" result is synchronous; a "task" result runs async and must be polled
    if result.get("kind") == "task":
        task_id = result["id"]
        state = result.get("status", {}).get("state")
        for _ in range(poll_attempts):
            if state in TERMINAL_A2A_STATES:
                break
            time.sleep(poll_delay)
            result = _jsonrpc(message_url, headers, "tasks/get", {"id": task_id})
            state = result.get("status", {}).get("state")
            utils.print_info(f"Task {task_id} state: {state}")

        if state != "completed":
            utils.print_error(f"A2A task ended in state '{state}':\n{json.dumps(result, indent=2)}")
            return

    answer = _extract_text(result)
    utils.print_ok(f"A2A response:\n{answer if answer else json.dumps(result, indent=2)}")

# Invoke the Weather agent over A2A with a simple discovery question
weather_a2a_apim_url = f"{apim_resource_gateway_url}/{weather_api_path}-a2a"
weather_a2a_apim_card = f"{weather_a2a_apim_url}/agent-card.json"
invoke_a2a_agent(weather_a2a_apim_url, weather_a2a_apim_card, "What can you help with?")


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.